# Legacy mode: the original package's assumptions, on request

wfdb-x never assumes a sampling frequency, gain, ADC resolution or units
that a record or the user does not give (see notebook 03). For
compatibility, the behavior of the original WFDB package, which assumed
250 Hz, 200 ADC units per millivolt, 12 bits and millivolts, remains
available on explicit request:

* the command-line tools accept `--legacy` as their first argument;
* the high-level functions accept `legacy=True`;
* `wfdbx.wfdbx_set_legacy(1)` turns the library's legacy mode on.

**Use it for compatibility only**: the values it assumes are exactly the
silent defaults wfdb-x removed, so results computed with them are wrong
whenever the assumption is.

In [ ]:
import numpy as np
import wfdbx

# Records are written to, and read from, the current directory (no input
# or output directory is set).

# A header that does not give the sampling frequency of its record
with open("nofreq.hea", "w") as f:
    f.write("nofreq 1\nnofreq.dat 16 200 16 0 0 0 0 sig\n")
with open("nofreq.dat", "wb") as f:
    f.write(np.arange(1000, dtype="<i2").tobytes())

In [ ]:
rec = wfdbx.read_record("nofreq")
print("sampling frequency:", rec.fs, "(not specified)")
try:
    wfdbx.read_record("nofreq", sampto="0:1")       # a time needs a frequency
except ValueError as err:
    print("ValueError:", err)

In [ ]:
old = wfdbx.read_record("nofreq", sampto="0:1", legacy=True)
print("legacy mode assumes", old.fs, "Hz:", len(old.signals), "samples in '1 second'")

The same with the tools: `wrsamp` without `-F`, `-G`, `-R` and `-U` is an
error, while `wrsamp --legacy` assumes the old values:

In [ ]:
import subprocess
import sys

text = "".join(f"{i}\n" for i in range(10))
for args in (["-o", "legacy1"], ["--legacy", "-o", "legacy1"]):
    r = subprocess.run([sys.executable, "-m", "wfdbx.cli", "wrsamp", *args],
                       input=text, capture_output=True, text=True)
    print("wrsamp", *args, "->", r.returncode, r.stderr.strip())
print(open("legacy1.hea").read())